
# 🚕 Taxi Fleet Performance Analysis  
### Operational & Driver Performance Intelligence

This notebook analyzes one week of taxi fleet activity to evaluate operational efficiency and driver performance.


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

sns.set_style("whitegrid")


## Data Loading & Cleaning

In [ ]:

df = pd.read_csv("Trips_activity.csv")

df = df.drop(columns=["Unnamed: 0"], errors="ignore")

datetime_cols = ["solicitud_viaje", "final_viaje", "inicio_turno"]
for col in datetime_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

df.head()


## Completed Trips per Day

In [ ]:

df_completed = df[df["viaje_estado"] == "completed"].copy()

df_completed["trip_date"] = df_completed["final_viaje"].dt.date

trips_per_day = df_completed.groupby("trip_date").size()

plt.figure(figsize=(10,6))
trips_per_day.plot(kind="bar")
plt.title("Completed Trips per Day")
plt.ylabel("Number of Trips")
plt.xticks(rotation=45)
plt.show()

trips_per_day.sort_values(ascending=False)


## Revenue by Hour

In [ ]:

df_completed["hour"] = df_completed["final_viaje"].dt.hour

df_filtered = df_completed[
    (df_completed["viaje_precio"] > 0) &
    (df_completed["viaje_precio"] < 250)
]

revenue_per_hour = df_filtered.groupby("hour")["viaje_precio"].sum()

plt.figure(figsize=(10,6))
revenue_per_hour.plot()
plt.title("Total Revenue per Hour")
plt.xlabel("Hour of Day")
plt.ylabel("Total Revenue")
plt.show()

revenue_per_hour.sort_values(ascending=False).head(3)


## Idle Time – Driver 673

In [ ]:

driver_673 = df_completed[df_completed["id_conductor"] == 673].copy()
driver_673 = driver_673.sort_values("solicitud_viaje")

driver_673["idle_time"] = (
    driver_673["solicitud_viaje"] -
    driver_673["final_viaje"].shift(1)
)

driver_673 = driver_673[
    (driver_673["idle_time"].dt.total_seconds() > 0) &
    (driver_673["idle_time"].dt.total_seconds() < 1800)
]

avg_idle_minutes = driver_673["idle_time"].dt.total_seconds().mean() / 60
round(avg_idle_minutes, 2)


## Driver Segmentation (K-Means)

In [ ]:

driver_summary = df_completed.groupby("id_conductor").agg(
    completed_trips=("viaje_precio", "count"),
    total_revenue=("viaje_precio", "sum"),
    total_km=("viaje_distancia", "sum")
).reset_index()

X = driver_summary[["completed_trips", "total_revenue", "total_km"]]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=20)
driver_summary["cluster"] = kmeans.fit_predict(X_scaled)

silhouette_score(X_scaled, driver_summary["cluster"])
